In [2]:
import scipy
import os
import pandas as pd
from tqdm import tqdm 
import numpy as np

In [ ]:
def find_indexWOW(arr):
    left, right = 0, len(arr) - 1
    idx = -1
    while left <= right:
        mid = left + (right - left) // 2
        if arr[mid] == 1:
            left = mid + 1
        elif arr[mid] == 0:
            if mid > 0 and arr[mid - 1] == 1:
                idx = mid
                break
            if arr[:mid].sum() == 0:
                left = mid + 1
            else:
                right = mid - 1
    
    if arr[idx:].sum() == 0:
        return idx
    return find_indexWOW(arr[idx+1:]) + idx + 1

complete_folder = "C:/Users/HP/Desktop/ML/project"
flights , timestamps, indices, latitude, longitude, magneticHeading = [], [], [] , [] , [], []
for folder in os.listdir(complete_folder):
    if 'Tail' not in folder:
        continue
    extractedMATfiles = os.path.join(complete_folder, folder)
    for idx, timestamp in enumerate(os.listdir(extractedMATfiles)):
            # find all files where sum(WOW) > 0
            try:
                data = scipy.io.loadmat(extractedMATfiles + "/" + timestamp)
            except:
                continue
            wow_data = data['WOW'][0][0][0]
            if wow_data.sum() > 0:
                wow_data = data['WOW'][0][0][0].reshape(-1)
                longitude_data = data['LONP'][0][0][0].reshape(-1)
                latitude_data = data['LATP'][0][0][0].reshape(-1)
                magneticHeading_data = data['MH'][0][0][0].reshape(-1)
                touchdown_idx = find_indexWOW(wow_data)
                magneticHeadingLanding = magneticHeading_data[touchdown_idx*4 - 4:touchdown_idx*4 + 4]
                flights.append("666_8")
                timestamps.append(timestamp)
                indices.append(touchdown_idx)
                latitude.append(latitude_data[touchdown_idx])
                longitude.append(longitude_data[touchdown_idx])
                magneticHeading.append(magneticHeadingLanding)
df = pd.DataFrame({'Flights' : flights , 'Timestamps': timestamps, 'Touchdown_idx': indices , 'Latitude': latitude, 'Longitude': longitude, 'MagneticHeading': magneticHeading})
df.to_csv("MagneticHeadingDataLanding.csv", index=False)

In [3]:
def find_indexN1(arr):
    left = len(arr) // 2
    right = len(arr) - 1
    result = -1
    
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] < 40:
            if all(x < 40 for x in arr[mid:]):
                result = mid 
                right = mid - 1  # Try to find an earlier valid point
            else:
                left = mid + 1
        else:
            left = mid + 1

    return result
        
completeFolder = "C:/Users/HP/Desktop/ML/project"
flights, timestamps, indices, latitude, longitude, magneticHeading = [], [], [], [], [], []

for folder in os.listdir(completeFolder):
    if 'Tail' not in folder:
        continue
    extractedMatFiles = os.path.join(completeFolder, folder)
    for idx, timestamp in enumerate(os.listdir(extractedMatFiles)):
        # Find all files where sum(wow) > 0
        try:
            data = scipy.io.loadmat(extractedMatFiles + "/" + timestamp)
        except:
            continue
        
        n1_1Data = data['N1_1'][0][0][0].reshape(-1)
        wowData = data['WOW'][0][0][0]
        touchdownIdx = find_indexN1(n1_1Data)
        
        if wowData.sum() > 0 and touchdownIdx != -1:
            flightName = folder.split('_')[1] + '_' + folder.split('_')[2]
            longitudeData = data['LONP'][0][0][0].reshape(-1)
            latitudeData = data['LATP'][0][0][0].reshape(-1)
            magneticHeadingData = data['MH'][0][0][0].reshape(-1)
            
            flights.append(flightName)
            timestamps.append(timestamp)
            indices.append(touchdownIdx)
            latitude.append(latitudeData[touchdownIdx // 4])
            longitude.append(longitudeData[touchdownIdx // 4])
            magneticHeading.append(magneticHeadingData[touchdownIdx])

df = pd.DataFrame({
    'Flights': flights,
    'Timestamps': timestamps,
    'TouchdownIdx': indices,
    'Latitude': latitude,
    'Longitude': longitude,
    'MagneticHeading': magneticHeading
})

df.to_csv("N1ApproachTouchdownCoordinates.csv", index=False)

In [2]:
str = "Tail_666_8"
str.split('_')[1] + '_' + str.split('_')[2]

'666_8'

In [5]:
import zipfile

# Path to the zip file
zip_path = 'C:/Users/HP/Desktop/ML/project/HoneyWell-Analysis-main.zip'

# Open the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    # List all files in the zip
    print(zip_ref.namelist())

    # Read a specific file's contents
    with zip_ref.open('filename_inside_zip.txt') as file:
        content = file.read().decode('utf-8')
        print(content)


['HoneyWell-Analysis-main/', 'HoneyWell-Analysis-main/Earlier Work/', 'HoneyWell-Analysis-main/Earlier Work/.DS_Store', 'HoneyWell-Analysis-main/Earlier Work/.DS_Store:Zone.Identifier', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/.DS_Store', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/.DS_Store:Zone.Identifier', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/Airports_max_LongLandings_absolutefrequency.csv', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/Airports_max_LongLandings_absolutefrequency.csv:Zone.Identifier', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/Airports_max_LongLandings_relativeFreq.csv', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/Airports_max_LongLandings_relativeFreq.csv:Zone.Identifier', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/FlightPathVIsualization.ipynb', 'HoneyWell-Analysis-main/Earlier Work/Initial Work/FlightPathVIsualization.ipynb:Zone.Identifier', 'HoneyWell

KeyError: "There is no item named 'filename_inside_zip.txt' in the archive"